In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
from matplotlib import pyplot as plt
from sklearn.model_selection import cross_val_score

from sklearn import metrics
from collections import Counter

Read from local

In [ ]:
#df = pd.read_excel(r'E:\Users\Avishka\LECTURE_MATERIALS\semester 7\FYP\ML\data\dataset.xlsx')

C:\Users\Avishka Koswaththa\AppData\Roaming\Python\Python312\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
column_list = df.columns.tolist()
print(column_list)

['PRODUCT_CODE', 'PRODUCT_NAME', 'PRODUCT_CATEGORY', 'CONTRACT_NO', 'CONTRACT_STATUS', 'CONTRACT_DATE', 'RECOVERY_STATUS', 'LAST_PAYMENT_DATE', 'RE_PROCESS_DATE', 'RESCHEDULE', 'DUE_FREQUENCY', 'NET_RENTAL', 'NO_OF_RENTAL', 'PAID_RENTALS', 'CB_ARREARS_AGE', 'ASSET_TYPE_NAME', 'YOM', 'MAKE', 'MODEL_NAME', 'REGISTRATION', 'REGISTRATION_NO', 'GENDER', 'CITY', 'DISTRICT_NAME', 'PROVINCE_NAME', 'FINANCE_AMOUNT', 'CUSTOMER_VALUATION', 'EFFECTIVE_RATE', 'AGE', 'MARITAL_STATUS', 'INCOME', 'EXPENSE']


In [4]:

status_map = {
    'S': 'Non-Default',
    'O': 'Non-Default',
    'RL': 'Non-Default',  
    'W': 'Default',
    'R': 'Default'
}


df['CONTRACT_STATUS'] = df['CONTRACT_STATUS'].map(status_map)


df['CONTRACT_STATUS'] = df['CONTRACT_STATUS'].astype('category')


In [5]:
import plotly.express as px


counts = df['CONTRACT_STATUS'].value_counts()


counts = counts.reindex(["Non-Default", "Default"])

fig = px.pie(
    values=counts.values,
    names=counts.index,
    width=700,
    height=400,
    color_discrete_sequence=["skyblue", "black"],
    title="Default vs Non-Default"
)

fig.show()


In [6]:
df.duplicated().any()


True

In [7]:
df = df.drop_duplicates()


In [8]:
df.isnull().sum()

PRODUCT_CODE              0
PRODUCT_NAME              0
PRODUCT_CATEGORY          0
CONTRACT_NO               0
CONTRACT_STATUS           0
CONTRACT_DATE             0
RECOVERY_STATUS       44957
LAST_PAYMENT_DATE        50
RE_PROCESS_DATE       88515
RESCHEDULE                0
DUE_FREQUENCY             0
NET_RENTAL                0
NO_OF_RENTAL              0
PAID_RENTALS              0
CB_ARREARS_AGE            0
ASSET_TYPE_NAME           0
YOM                       0
MAKE                      0
MODEL_NAME               12
REGISTRATION              0
REGISTRATION_NO         613
GENDER                  467
CITY                      0
DISTRICT_NAME             0
PROVINCE_NAME             0
FINANCE_AMOUNT            0
CUSTOMER_VALUATION      299
EFFECTIVE_RATE            0
AGE                      78
MARITAL_STATUS         2233
INCOME                 1352
EXPENSE                 744
dtype: int64

In [9]:
df.fillna(df.median(numeric_only=True), inplace=True)  # median for numeric columns

for col in df.select_dtypes(include=['object', 'category']).columns:
    df[col].fillna(df[col].mode()[0], inplace=True)    # mode for categorical columns


In [10]:
df.isnull().sum()

PRODUCT_CODE          0
PRODUCT_NAME          0
PRODUCT_CATEGORY      0
CONTRACT_NO           0
CONTRACT_STATUS       0
CONTRACT_DATE         0
RECOVERY_STATUS       0
LAST_PAYMENT_DATE     0
RE_PROCESS_DATE       0
RESCHEDULE            0
DUE_FREQUENCY         0
NET_RENTAL            0
NO_OF_RENTAL          0
PAID_RENTALS          0
CB_ARREARS_AGE        0
ASSET_TYPE_NAME       0
YOM                   0
MAKE                  0
MODEL_NAME            0
REGISTRATION          0
REGISTRATION_NO       0
GENDER                0
CITY                  0
DISTRICT_NAME         0
PROVINCE_NAME         0
FINANCE_AMOUNT        0
CUSTOMER_VALUATION    0
EFFECTIVE_RATE        0
AGE                   0
MARITAL_STATUS        0
INCOME                0
EXPENSE               0
dtype: int64

In [11]:
cols_to_drop = [
    'CONTRACT_NO', 'PRODUCT_CODE', 'PRODUCT_NAME', 'PRODUCT_CATEGORY', 
    'MAKE', 'MODEL_NAME', 'REGISTRATION', 'REGISTRATION_NO', 'GENDER',
    'CITY', 'DISTRICT_NAME', 'PROVINCE_NAME', 'CONTRACT_DATE', 'RE_PROCESS_DATE', 'YOM', 'ASSET_TYPE_NAME',
    'PAID_RENTALS', 'DAYS_SINCE_LAST_PAYMENT', 'CB_ARREARS_AGE', 'RECOVERY_STATUS_FREQ_ENC', 'RESCHEDULE',
    'RECOVERY_STATUS', 'LAST_PAYMENT_DATE'
]


df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])


In [12]:
column_list = df.columns.tolist()
print(column_list)

['CONTRACT_STATUS', 'DUE_FREQUENCY', 'NET_RENTAL', 'NO_OF_RENTAL', 'FINANCE_AMOUNT', 'CUSTOMER_VALUATION', 'EFFECTIVE_RATE', 'AGE', 'MARITAL_STATUS', 'INCOME', 'EXPENSE']


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99326 entries, 0 to 99999
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   CONTRACT_STATUS     99326 non-null  category
 1   DUE_FREQUENCY       99326 non-null  object  
 2   NET_RENTAL          99326 non-null  float64 
 3   NO_OF_RENTAL        99326 non-null  float64 
 4   FINANCE_AMOUNT      99326 non-null  float64 
 5   CUSTOMER_VALUATION  99326 non-null  float64 
 6   EFFECTIVE_RATE      99326 non-null  float64 
 7   AGE                 99326 non-null  float64 
 8   MARITAL_STATUS      99326 non-null  object  
 9   INCOME              99326 non-null  float64 
 10  EXPENSE             99326 non-null  float64 
dtypes: category(1), float64(8), object(2)
memory usage: 8.4+ MB


In [14]:
for col in [ 'MARITAL_STATUS']:
    print(f"Unique values in '{col}':")
    print(df[col].unique())
    print()


Unique values in 'MARITAL_STATUS':
['M' 'S' 'W' 'D' 'O' 'U']



In [15]:


# One-hot encode MARITAL_STATUS 
df['MARITAL_STATUS'] = df['MARITAL_STATUS'].replace({'O': 'Other_Unknown', 'U': 'Other_Unknown'})
df = pd.get_dummies(df, columns=['MARITAL_STATUS'], drop_first=True)



df.head()


,CONTRACT_STATUS,DUE_FREQUENCY,NET_RENTAL,NO_OF_RENTAL,FINANCE_AMOUNT,CUSTOMER_VALUATION,EFFECTIVE_RATE,AGE,INCOME,EXPENSE,MARITAL_STATUS_M,MARITAL_STATUS_Other_Unknown,MARITAL_STATUS_S,MARITAL_STATUS_W
0,Non-Default,M,11996.0,36.0,290000.0,375000.0,28.00,62.0,46000.0,15000.0,True,False,False,False
1,Default,M,13070.0,48.0,375000.0,475000.0,28.00,40.0,40000.0,10750.0,True,False,False,False
2,Non-Default,M,12410.0,36.0,300000.0,500000.0,28.01,50.0,150000.0,75000.0,False,False,True,False
3,Non-Default,M,11108.0,48.0,365000.0,600000.0,20.00,67.0,35000.0,9000.0,True,False,False,False
4,Non-Default,M,19169.0,48.0,550000.0,700000.0,28.00,35.0,80000.0,30000.0,False,False,True,False


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99326 entries, 0 to 99999
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   CONTRACT_STATUS               99326 non-null  category
 1   DUE_FREQUENCY                 99326 non-null  object  
 2   NET_RENTAL                    99326 non-null  float64 
 3   NO_OF_RENTAL                  99326 non-null  float64 
 4   FINANCE_AMOUNT                99326 non-null  float64 
 5   CUSTOMER_VALUATION            99326 non-null  float64 
 6   EFFECTIVE_RATE                99326 non-null  float64 
 7   AGE                           99326 non-null  float64 
 8   INCOME                        99326 non-null  float64 
 9   EXPENSE                       99326 non-null  float64 
 10  MARITAL_STATUS_M              99326 non-null  bool    
 11  MARITAL_STATUS_Other_Unknown  99326 non-null  bool    
 12  MARITAL_STATUS_S              99326 non-null  bool 

In [17]:
column_list = df.columns.tolist()
print(column_list)

['CONTRACT_STATUS', 'DUE_FREQUENCY', 'NET_RENTAL', 'NO_OF_RENTAL', 'FINANCE_AMOUNT', 'CUSTOMER_VALUATION', 'EFFECTIVE_RATE', 'AGE', 'INCOME', 'EXPENSE', 'MARITAL_STATUS_M', 'MARITAL_STATUS_Other_Unknown', 'MARITAL_STATUS_S', 'MARITAL_STATUS_W']


In [18]:
df['AGE'] = df['AGE'].clip(lower=18, upper=85)

In [19]:
df.describe()

,NET_RENTAL,NO_OF_RENTAL,FINANCE_AMOUNT,CUSTOMER_VALUATION,EFFECTIVE_RATE,AGE,INCOME,EXPENSE
count,9.932600e+04,99326.000000,9.932600e+04,9.932600e+04,99326.000000,99326.000000,9.932600e+04,9.932600e+04
mean,1.529106e+04,36.673740,3.742441e+05,5.137963e+05,28.700440,40.197370,1.008061e+08,1.012160e+07
std,3.610578e+04,11.044738,5.781489e+05,8.072094e+05,6.943737,10.622112,3.172989e+10,3.172989e+09
min,0.000000e+00,1.000000,9.729130e+03,0.000000e+00,0.000000,18.000000,3.000000e-01,1.000000e+00
25%,7.000250e+03,36.000000,1.522500e+05,2.069900e+05,25.022500,32.000000,3.200000e+04,9.500000e+03
50%,9.412000e+03,36.000000,2.055000e+05,2.785710e+05,30.000000,39.000000,5.000000e+04,1.500000e+04
75%,1.363200e+04,42.000000,3.290000e+05,4.500000e+05,32.000000,47.000000,9.000000e+04,3.000000e+04
max,3.200000e+06,242.810000,3.200000e+07,5.450000e+07,221.570000,85.000000,1.000000e+13,1.000000e+12


In [20]:
def treat_outliers(df, col):
    '''
    Treats outliers in a numerical variable using IQR method.
    Returns the modified DataFrame.
    '''
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    Lower_Whisker = Q1 - 1.5 * IQR
    Upper_Whisker = Q3 + 1.5 * IQR

    # Count how many values are clipped
    n_clipped = ((df[col] < Lower_Whisker) | (df[col] > Upper_Whisker)).sum()
    print(f"{col}: {n_clipped} values clipped")

    # Clip the values
    df[col] = np.clip(df[col], Lower_Whisker, Upper_Whisker)
    return df


def treat_outliers_all(df, col_list):
    '''
    Treats outliers in all numerical variables in the provided list.
    '''
    
    for col in col_list:
        df = treat_outliers(df, col)
    return df


In [21]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
exclude_cols = [ 'DUE_FREQUENCY', 'MARITAL_STATUS_*']  # exclude binary/numeric categorical columns
cols_to_treat = [col for col in numerical_cols if col not in exclude_cols]

df = treat_outliers_all(df, cols_to_treat)


NET_RENTAL: 12333 values clipped
NO_OF_RENTAL: 29245 values clipped
FINANCE_AMOUNT: 13427 values clipped
CUSTOMER_VALUATION: 13921 values clipped
EFFECTIVE_RATE: 3900 values clipped
AGE: 455 values clipped
INCOME: 9059 values clipped
EXPENSE: 9203 values clipped


In [22]:
df.describe()

,NET_RENTAL,NO_OF_RENTAL,FINANCE_AMOUNT,CUSTOMER_VALUATION,EFFECTIVE_RATE,AGE,INCOME,EXPENSE
count,99326.000000,99326.000000,99326.000000,99326.000000,99326.000000,99326.000000,99326.000000,99326.000000
mean,11325.586289,37.169981,267479.072775,362683.293797,28.637053,40.181690,68791.471990,21930.506927
std,5967.582739,7.648561,161440.552926,227901.077819,6.122859,10.574171,48132.517802,17366.562175
min,0.000000,27.000000,9729.130000,0.000000,14.556250,18.000000,0.300000,1.000000
25%,7000.250000,36.000000,152250.000000,206990.000000,25.022500,32.000000,32000.000000,9500.000000
50%,9412.000000,36.000000,205500.000000,278571.000000,30.000000,39.000000,50000.000000,15000.000000
75%,13632.000000,42.000000,329000.000000,450000.000000,32.000000,47.000000,90000.000000,30000.000000
max,23579.625000,51.000000,594125.000000,814515.000000,42.466250,69.500000,177000.000000,60750.000000


In [23]:
import plotly.express as px


counts = df['CONTRACT_STATUS'].value_counts()


counts = counts.reindex(["Non-Default", "Default"])

fig = px.pie(
    values=counts.values,
    names=counts.index,
    width=700,
    height=400,
    color_discrete_sequence=["skyblue", "black"],
    title="Default vs Non-Default"
)

fig.show()


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99326 entries, 0 to 99999
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   CONTRACT_STATUS               99326 non-null  category
 1   DUE_FREQUENCY                 99326 non-null  object  
 2   NET_RENTAL                    99326 non-null  float64 
 3   NO_OF_RENTAL                  99326 non-null  float64 
 4   FINANCE_AMOUNT                99326 non-null  float64 
 5   CUSTOMER_VALUATION            99326 non-null  float64 
 6   EFFECTIVE_RATE                99326 non-null  float64 
 7   AGE                           99326 non-null  float64 
 8   INCOME                        99326 non-null  float64 
 9   EXPENSE                       99326 non-null  float64 
 10  MARITAL_STATUS_M              99326 non-null  bool    
 11  MARITAL_STATUS_Other_Unknown  99326 non-null  bool    
 12  MARITAL_STATUS_S              99326 non-null  bool 

In [25]:
df['CONTRACT_STATUS'] = df['CONTRACT_STATUS'].map({'Non-Default': 0, 'Default': 1})


In [27]:
df.head()

,CONTRACT_STATUS,DUE_FREQUENCY,NET_RENTAL,NO_OF_RENTAL,FINANCE_AMOUNT,CUSTOMER_VALUATION,EFFECTIVE_RATE,AGE,INCOME,EXPENSE,MARITAL_STATUS_M,MARITAL_STATUS_Other_Unknown,MARITAL_STATUS_S,MARITAL_STATUS_W
0,0,M,11996.0,36.0,290000.0,375000.0,28.00,62.0,46000.0,15000.0,True,False,False,False
1,1,M,13070.0,48.0,375000.0,475000.0,28.00,40.0,40000.0,10750.0,True,False,False,False
2,0,M,12410.0,36.0,300000.0,500000.0,28.01,50.0,150000.0,60750.0,False,False,True,False
3,0,M,11108.0,48.0,365000.0,600000.0,20.00,67.0,35000.0,9000.0,True,False,False,False
4,0,M,19169.0,48.0,550000.0,700000.0,28.00,35.0,80000.0,30000.0,False,False,True,False


In [28]:
for col in [ 'DUE_FREQUENCY']:
    print(f"Unique values in '{col}':")
    print(df[col].unique())
    print()


Unique values in 'DUE_FREQUENCY':
['M' 'D']



In [29]:
df['DUE_FREQUENCY'].value_counts()

DUE_FREQUENCY
M    99324
D        2
Name: count, dtype: int64

In [30]:
df['DUE_FREQUENCY'] = df['DUE_FREQUENCY'].map({'M': 0, 'D': 1})


In [31]:
df.head()

,CONTRACT_STATUS,DUE_FREQUENCY,NET_RENTAL,NO_OF_RENTAL,FINANCE_AMOUNT,CUSTOMER_VALUATION,EFFECTIVE_RATE,AGE,INCOME,EXPENSE,MARITAL_STATUS_M,MARITAL_STATUS_Other_Unknown,MARITAL_STATUS_S,MARITAL_STATUS_W
0,0,0,11996.0,36.0,290000.0,375000.0,28.00,62.0,46000.0,15000.0,True,False,False,False
1,1,0,13070.0,48.0,375000.0,475000.0,28.00,40.0,40000.0,10750.0,True,False,False,False
2,0,0,12410.0,36.0,300000.0,500000.0,28.01,50.0,150000.0,60750.0,False,False,True,False
3,0,0,11108.0,48.0,365000.0,600000.0,20.00,67.0,35000.0,9000.0,True,False,False,False
4,0,0,19169.0,48.0,550000.0,700000.0,28.00,35.0,80000.0,30000.0,False,False,True,False


In [33]:
# Convert boolean one-hot columns to float
marital_cols = ['MARITAL_STATUS_M', 'MARITAL_STATUS_Other_Unknown',
                'MARITAL_STATUS_S', 'MARITAL_STATUS_W']
df[marital_cols] = df[marital_cols].astype(float)


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99326 entries, 0 to 99999
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype   
---  ------                        --------------  -----   
 0   CONTRACT_STATUS               99326 non-null  category
 1   DUE_FREQUENCY                 99326 non-null  int64   
 2   NET_RENTAL                    99326 non-null  float64 
 3   NO_OF_RENTAL                  99326 non-null  float64 
 4   FINANCE_AMOUNT                99326 non-null  float64 
 5   CUSTOMER_VALUATION            99326 non-null  float64 
 6   EFFECTIVE_RATE                99326 non-null  float64 
 7   AGE                           99326 non-null  float64 
 8   INCOME                        99326 non-null  float64 
 9   EXPENSE                       99326 non-null  float64 
 10  MARITAL_STATUS_M              99326 non-null  float64 
 11  MARITAL_STATUS_Other_Unknown  99326 non-null  float64 
 12  MARITAL_STATUS_S              99326 non-null  float

In [35]:
df.head()

,CONTRACT_STATUS,DUE_FREQUENCY,NET_RENTAL,NO_OF_RENTAL,FINANCE_AMOUNT,CUSTOMER_VALUATION,EFFECTIVE_RATE,AGE,INCOME,EXPENSE,MARITAL_STATUS_M,MARITAL_STATUS_Other_Unknown,MARITAL_STATUS_S,MARITAL_STATUS_W
0,0,0,11996.0,36.0,290000.0,375000.0,28.00,62.0,46000.0,15000.0,1.0,0.0,0.0,0.0
1,1,0,13070.0,48.0,375000.0,475000.0,28.00,40.0,40000.0,10750.0,1.0,0.0,0.0,0.0
2,0,0,12410.0,36.0,300000.0,500000.0,28.01,50.0,150000.0,60750.0,0.0,0.0,1.0,0.0
3,0,0,11108.0,48.0,365000.0,600000.0,20.00,67.0,35000.0,9000.0,1.0,0.0,0.0,0.0
4,0,0,19169.0,48.0,550000.0,700000.0,28.00,35.0,80000.0,30000.0,0.0,0.0,1.0,0.0


Save to local

In [ ]:
#df.to_csv(r'E:\Users\Avishka\LECTURE_MATERIALS\semester 7\FYP\ML\data\new_cleaned_dataset5.csv', index=False)
